In [1]:
from pathlib import Path

base = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data")
splits = ["train", "val", "test"]

def count_instances(lbl_files):
    total = 0
    for p in lbl_files:
        with open(p, "r") as f:
            total += sum(1 for ln in f if ln.strip())
    return total

def stemset(paths):
    return {p.stem for p in paths}

grand_imgs = grand_lbls = grand_insts = 0
grand_pos = grand_neg = 0

print("===== Dataset Summary by Region & Split =====")
for region_dir in sorted([d for d in base.iterdir() if d.is_dir()]):
    region = region_dir.name
    reg_imgs = reg_lbls = reg_insts = 0
    reg_pos = reg_neg = 0
    print(f"\n[Region] {region}")
    for split in splits:
        img_dir = region_dir / split / "images"
        lbl_dir = region_dir / split / "labels"

        img_files = list(img_dir.glob("*.png")) if img_dir.exists() else []
        lbl_files = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

        inst_cnt = count_instances(lbl_files)

        img_stems = stemset(img_files)
        lbl_stems = stemset(lbl_files)

        pos_imgs = len(img_stems & lbl_stems)  # positive = with labels
        neg_imgs = len(img_stems - lbl_stems)  # negative = without labels

        print(f"  {split.capitalize():5} -> Images: {len(img_files):5} | Labels: {len(lbl_files):5} | "
              f"Instances: {inst_cnt:6} | Pos: {pos_imgs:5} | Neg: {neg_imgs:5}")

        reg_imgs += len(img_files)
        reg_lbls += len(lbl_files)
        reg_insts += inst_cnt
        reg_pos  += pos_imgs
        reg_neg  += neg_imgs

    print(f"  -- Region totals -> Images: {reg_imgs} | Labels: {reg_lbls} | Instances: {reg_insts} "
          f"| Pos: {reg_pos} | Neg: {reg_neg}")

    grand_imgs += reg_imgs
    grand_lbls += reg_lbls
    grand_insts += reg_insts
    grand_pos  += reg_pos
    grand_neg  += reg_neg

print("\n===== GRAND TOTALS =====")
print(f"Images: {grand_imgs} | Labels: {grand_lbls} | Instances: {grand_insts} | Pos: {grand_pos} | Neg: {grand_neg}")

===== Dataset Summary by Region & Split =====

[Region] bangladesh
  Train -> Images:  7672 | Labels:  5634 | Instances:   8051 | Pos:  5634 | Neg:  2038
  Val   -> Images:  1631 | Labels:   612 | Instances:    675 | Pos:   612 | Neg:  1019
  Test  -> Images:  1633 | Labels:   612 | Instances:    700 | Pos:   612 | Neg:  1021
  -- Region totals -> Images: 10936 | Labels: 6858 | Instances: 9426 | Pos: 6858 | Neg: 4078

[Region] pak_punjab
  Train -> Images:  8585 | Labels:  6364 | Instances:   8632 | Pos:  6364 | Neg:  2221
  Val   -> Images:  2873 | Labels:   737 | Instances:    838 | Pos:   737 | Neg:  2136
  Test  -> Images:  2892 | Labels:   738 | Instances:    833 | Pos:   738 | Neg:  2154
  -- Region totals -> Images: 14350 | Labels: 7839 | Instances: 10303 | Pos: 7839 | Neg: 6511

[Region] uttar_pradesh
  Train -> Images:  9122 | Labels:  6940 | Instances:   9351 | Pos:  6940 | Neg:  2182
  Val   -> Images:  2012 | Labels:   921 | Instances:    980 | Pos:   921 | Neg:  1091
  Tes

In [3]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point

# =======================
# CONFIG
# =======================
BASE = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data")
PATCH_META_DIR = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/patch_metadata")

META_FILES = {
    "uttar_pradesh": PATCH_META_DIR / "uttar_pradesh_patch_centers.geojson",
    "pak_punjab":    PATCH_META_DIR / "pak_punjab_metadata.geojson",
    "bangladesh":    PATCH_META_DIR / "bangladesh_metadata.geojson",
}

SPLITS = ["train", "val", "test"]
IMG_EXT = ".png"  # your patch images

# =======================
# Helpers
# =======================
def parse_patch_id(stem: str):
    """
    Parse 'lat_lon' from filename stem like '25.2879_80.4283' -> (lat, lon)
    """
    lat_str, lon_str = stem.split("_")
    return round(float(lat_str), 4), round(float(lon_str), 4)

def load_all_metadata() -> gpd.GeoDataFrame:
    """
    Read all state metadata, ensure WGS84, and add rounding cols:
    lat4, lon4 that match your filename precision (.4f).
    """
    frames = []
    for state, path in META_FILES.items():
        gdf = gpd.read_file(path)
        # Ensure CRS
        if gdf.crs is None:
            gdf.set_crs(epsg=4326, inplace=True)
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(4326)

        # We expect lat_center/lon_center in the metadata you produced
        if not {"lat_center", "lon_center"}.issubset(gdf.columns):
            raise ValueError(f"{path} must have 'lat_center' and 'lon_center' columns.")

        gdf["lat4"] = gdf["lat_center"].round(4)
        gdf["lon4"] = gdf["lon_center"].round(4)
        gdf["state"] = state
        frames.append(gdf[["state", "lat_center", "lon_center", "lat4", "lon4", "geometry"]])

    all_meta = pd.concat(frames, ignore_index=True)
    return gpd.GeoDataFrame(all_meta, geometry="geometry", crs="EPSG:4326")

def build_index(meta_gdf: gpd.GeoDataFrame):
    """
    Build a dictionary {(state, lat4, lon4) -> geometry}
    """
    idx = {}
    for row in meta_gdf.itertuples(index=False):
        idx[(row.state, row.lat4, row.lon4)] = row.geometry
    return idx

def image_iter(base: Path):
    """
    Yield tuples (state, split, img_path, patch_id, lat4, lon4)
    for every .png image under final_data/<state>/<split>/images
    """
    for state_dir in sorted([d for d in base.iterdir() if d.is_dir()]):
        state = state_dir.name  # 'uttar_pradesh' / 'pak_punjab' / 'bangladesh'
        for split in SPLITS:
            img_dir = state_dir / split / "images"
            if not img_dir.exists():
                continue
            for p in img_dir.glob(f"*{IMG_EXT}"):
                stem = p.stem
                try:
                    lat4, lon4 = parse_patch_id(stem)
                except Exception:
                    # Skip files that don't match 'lat_lon.png'
                    continue
                patch_id = f"{lat4:.4f}_{lon4:.4f}"
                yield state, split, p, patch_id, lat4, lon4

def build_patch_index_gdf() -> gpd.GeoDataFrame:
    """
    Main builder: returns GeoDataFrame with [state, split, patch_id, geometry]
    """
    meta = load_all_metadata()
    idx = build_index(meta)

    records = []
    missing = 0

    for state, split, path, patch_id, lat4, lon4 in image_iter(BASE):
        geom = idx.get((state, lat4, lon4))
        if geom is None:
            missing += 1
        records.append({
            "state": state,
            "split": split,
            "patch_id": patch_id,
            "geometry": geom
        })

    if missing:
        print(f"⚠️  {missing} images had no geometry match (state + lat4/lon4).")

    gdf = gpd.GeoDataFrame(pd.DataFrame.from_records(records), geometry="geometry", crs="EPSG:4326")
    return gdf

def get_geometry_for_image(img_path: str | Path, meta_index=None):
    """
    Fetch the polygon geometry for a single image path like:
    /.../final_data/uttar_pradesh/train/images/25.2879_80.4283.png

    Usage:
        geom = get_geometry_for_image("/full/path/to/25.2879_80.4283.png")
    """
    img_path = Path(img_path)
    state = img_path.parents[2].name  # .../<state>/<split>/images/<file>
    stem = img_path.stem
    lat4, lon4 = parse_patch_id(stem)

    if meta_index is None:
        meta_gdf = load_all_metadata()
        meta_index = build_index(meta_gdf)

    return meta_index.get((state, lat4, lon4))  # shapely Polygon or None

# =======================
# Run
# =======================
if __name__ == "__main__":
    gdf = build_patch_index_gdf()
    out_geojson = BASE / "patch_index_from_latlon_names.geojson"
    out_csv    = BASE / "patch_index_from_latlon_names.csv"

    gdf.to_file(out_geojson, driver="GeoJSON")
    gdf_csv = gdf.copy()
    gdf_csv["geometry"] = gdf_csv["geometry"].apply(lambda g: g.wkt if g is not None else None)
    gdf_csv.to_csv(out_csv, index=False)

    print(f"✅ Saved:\n  {out_geojson}\n  {out_csv}")

    # Example single-lookup (your example path)
    example = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh/train/images/25.2879_80.4283.png"
    geom = get_geometry_for_image(example)
    print("Example geometry:", "FOUND" if geom is not None else "NOT FOUND")

/tmp/ipykernel_3714213/2233730093.py:144: UserWarning: Geometry column does not contain geometry.
  gdf_csv["geometry"] = gdf_csv["geometry"].apply(lambda g: g.wkt if g is not None else None)


✅ Saved:
  /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/patch_index_from_latlon_names.geojson
  /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/patch_index_from_latlon_names.csv
Example geometry: FOUND


In [2]:
#!/usr/bin/env python3
"""
Download AlphaEarth Foundations (64-band) embeddings ONLY for:
  state = 'uttar_pradesh', split = 'val'

Creates:
  final_data/uttar_pradesh/val/embeddings/<lat>_<lon>.tif

Inputs:
  - final_data/patch_index_from_latlon_names.geojson
    columns: state, split, patch_id, geometry (EPSG:4326)
"""

from pathlib import Path
import time
import math
import json
import requests

import ee
import geopandas as gpd
import rasterio

# =======================
# CONFIG
# =======================
DATA_ROOT = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data")
PATCH_INDEX = DATA_ROOT / "patch_index_from_latlon_names.geojson"  # at the root
TARGET_STATE = "uttar_pradesh"
TARGET_SPLIT = "val"

YEAR_START, YEAR_END = "2024-01-01", "2025-01-01"
SCALE_M = 10
BANDS_COLLECTION_ID = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"  # AlphaEarth Foundations
MAX_RETRIES = 3
SLEEP_BETWEEN = 0.5  # seconds

# =======================
# EE init
# =======================
def init_ee():
    try:
        ee.Initialize()
        return
    except Exception as e1:
        print("ee.Initialize() failed, trying appdefault:", e1)
    try:
        ee.Authenticate(auth_mode='appdefault')
        ee.Initialize()
        return
    except Exception as e2:
        print("appdefault failed, falling back to interactive:", e2)
        ee.Authenticate()
        ee.Initialize()

# =======================
# Helpers
# =======================
def utm_epsg_from_lon(lat: float, lon: float) -> str:
    zone = int(math.floor((lon + 180) / 6) + 1)
    return f"EPSG:{32600 + zone}" if lat >= 0 else f"EPSG:{32700 + zone}"

def polygon_to_ee_region(poly):
    coords = list(poly.exterior.coords)
    return [[float(x), float(y)] for (x, y) in coords]

def build_image():
    return (ee.ImageCollection(BANDS_COLLECTION_ID)
            .filterDate(YEAR_START, YEAR_END)
            .mosaic()
            .toFloat())

def download_patch(img, region_coords, crs_epsg, scale, timeout_s=600):
    params = {
        "region": json.dumps(region_coords),
        "scale": scale,
        "crs": crs_epsg,
        "filePerBand": False,
        "format": "GEO_TIFF",
    }
    url = img.getDownloadURL(params)
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, timeout=timeout_s)
            r.raise_for_status()
            return r.content
        except Exception as e:
            last_exc = e
            wait = min(2 ** attempt, 10)
            print(f"  ⚠️  download failed (attempt {attempt}/{MAX_RETRIES}): {e}; retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {MAX_RETRIES} attempts: {last_exc}")

def ensure_gdf():
    if not PATCH_INDEX.exists():
        raise FileNotFoundError(f"Patch index not found: {PATCH_INDEX}")
    gdf = gpd.read_file(PATCH_INDEX)
    if gdf.crs is None:
        gdf.set_crs(epsg=4326, inplace=True)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)

    needed = {"state", "split", "patch_id", "geometry"}
    missing = needed - set(gdf.columns)
    if missing:
        raise ValueError(f"Patch index missing columns: {missing}")

    gdf = gdf.dropna(subset=["geometry"]).reset_index(drop=True)
    # Filter ONLY uttar_pradesh / val
    gdf = gdf[(gdf["state"] == TARGET_STATE) & (gdf["split"] == TARGET_SPLIT)].reset_index(drop=True)
    return gdf

def save_tif_bytes(raw_bytes, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "wb") as f:
        f.write(raw_bytes)

# =======================
# Main
# =======================
def main():
    init_ee()
    gdf = ensure_gdf()
    img = build_image()

    total = len(gdf)
    print(f"[{TARGET_STATE}/{TARGET_SPLIT}] patches: {total}. Starting downloads…")

    out_dir = DATA_ROOT / TARGET_STATE / TARGET_SPLIT / "embeddings"
    out_dir.mkdir(parents=True, exist_ok=True)

    done = skipped = failed = 0

    for row in gdf.itertuples(index=False):
        patch_id = row.patch_id  # '<lat>_<lon>'
        out_path = out_dir / f"{patch_id}.tif"
        if out_path.exists():
            skipped += 1
            continue

        # Prefer lat/lon from filename; fallback to centroid
        try:
            lat_str, lon_str = patch_id.split("_")
            lat = float(lat_str); lon = float(lon_str)
        except Exception:
            lat = float(row.geometry.centroid.y)
            lon = float(row.geometry.centroid.x)

        crs_epsg = utm_epsg_from_lon(lat, lon)
        region_coords = polygon_to_ee_region(row.geometry)

        try:
            raw = download_patch(img, region_coords, crs_epsg, SCALE_M)
            save_tif_bytes(raw, out_path)
            # quick check
            with rasterio.open(out_path) as src:
                if src.count != 64:
                    print(f"  ⚠️  {out_path.name}: expected 64 bands, got {src.count}")
            done += 1
        except Exception as e:
            failed += 1
            print(f"  ❌  {patch_id}: {e}")

        time.sleep(SLEEP_BETWEEN)

    print(f"\n✅ Finished [{TARGET_STATE}/{TARGET_SPLIT}]  Saved: {done} | Skipped: {skipped} | Failed: {failed}")

if __name__ == "__main__":
    main()

[uttar_pradesh/val] patches: 2012. Starting downloads…

✅ Finished [uttar_pradesh/val]  Saved: 2012 | Skipped: 0 | Failed: 0


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Concatenate per-region CSVs (train/val) into combined all_train.csv and all_val.csv.

Input CSVs should have identical headers:
  region,filename,aef_npy

Usage:
  python concat_csvs.py
"""

import csv
from pathlib import Path

# --- Edit these paths ---
BASE = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data")

train_csvs = [
    BASE / "uttar_pradesh_train_per_image_aef.csv",
    BASE / "bangladesh_train_per_image_aef.csv",
    BASE / "pak_punjab_train_per_image_aef.csv",
]

val_csvs = [
    BASE / "uttar_pradesh_val_per_image_aef.csv",
    BASE / "bangladesh_val_per_image_aef.csv",
    BASE / "pak_punjab_val_per_image_aef.csv",
]

out_train = BASE / "all_train.csv"
out_val   = BASE / "all_val.csv"


def concat_csv(inputs, output):
    rows = []
    header = None
    for i, f in enumerate(inputs):
        if not Path(f).exists():
            print(f"[WARN] Missing: {f}")
            continue
        with open(f, "r") as infile:
            reader = csv.reader(infile)
            this_header = next(reader)
            if header is None:
                header = this_header
            elif this_header != header:
                raise ValueError(f"Header mismatch in {f}: {this_header} vs {header}")
            for row in reader:
                rows.append(row)
        print(f"[OK] Read {f} ({len(rows)} total rows so far)")
    if not header:
        raise RuntimeError("No valid inputs")
    with open(output, "w", newline="") as out:
        writer = csv.writer(out)
        writer.writerow(header)
        writer.writerows(rows)
    print(f"[DONE] Wrote {output} ({len(rows)} rows)")


if __name__ == "__main__":
    concat_csv(train_csvs, out_train)
    concat_csv(val_csvs, out_val)

[OK] Read /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh_train_per_image_aef.csv (9122 total rows so far)
[OK] Read /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/bangladesh_train_per_image_aef.csv (16794 total rows so far)
[OK] Read /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/pak_punjab_train_per_image_aef.csv (25379 total rows so far)
[DONE] Wrote /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/all_train.csv (25379 rows)
[OK] Read /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh_val_per_image_aef.csv (2012 total rows so far)
[OK] Read /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/bangladesh_val_per_image_aef.csv (3643 total rows so far)
[OK] Read /h